# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

`mlcroissant` exposes record sets via `dataset.record_sets`, with each record set having a unique `@id`. For each record set, fields (columns) are described with their own `@id` as well.

In [ ]:
# Print available Record Sets and their Fields, referencing them by @id
print("Available Record Sets (by @id):\n-------------------------------")
record_sets = list(dataset.record_sets)

for rs in record_sets:
    print(f"Record Set @id: {rs['@id']}")
    print(f"  Name: {rs['name'] if 'name' in rs else rs.get('@id', '')}")
    print("  Fields (by @id):")
    for field in rs['fields']:
        fid = field.get('@id')
        fname = field.get('name', '')
        ftype = field.get('dataType', '')
        print(f"    - {fid} (name: {fname}, type: {ftype})")
    print("-------------------------------")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

Below, we demonstrate extracting all available record sets.

In [ ]:
# Extract data from all record sets
dataframes = {}
for rs in dataset.record_sets:
    record_set_id = rs['@id']
    print(f"Loading records for Record Set @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if len(records) == 0:
        print(f"  [Empty or non-tabular Record Set: {record_set_id}]")
        continue
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  Columns: {df.columns.tolist()}")
    print(f"  Sample records:\n{df.head()}\n---\n")

# For analysis, select a record set if at least one is loaded
if dataframes:
    chosen_record_set_id = list(dataframes.keys())[0]
    print(f"Using Record Set '@id' for further analysis: {chosen_record_set_id}")
else:
    chosen_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming data distributions, or grouping data by key attributes to prepare for further analysis.

We demonstrate EDA using the first available numeric field from the chosen record set (fields are referenced by their `@id`).

In [ ]:
import numpy as np

if chosen_record_set_id is not None and chosen_record_set_id in dataframes:
    df = dataframes[chosen_record_set_id]
    # Get the list of fields for this record set
    rs_metadata = next(rs for rs in dataset.record_sets if rs['@id'] == chosen_record_set_id)
    numeric_fields = [f['@id'] for f in rs_metadata['fields'] if f.get('dataType', '').lower() in ['number', 'float', 'integer']]
    group_fields = [f['@id'] for f in rs_metadata['fields'] if f.get('dataType', '').lower() in ['string', 'text']]

    if numeric_fields:
        numeric_field = numeric_fields[0]
        print(f"Using numeric field for filtering and normalization: {numeric_field}")
    else:
        numeric_field = df.select_dtypes(include=np.number).columns[0] if not df.select_dtypes(include=np.number).empty else None
        print(f"Auto-detected numeric field: {numeric_field}")

    if numeric_field and numeric_field in df.columns:
        # Attempt numeric conversion (if not already)
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

        threshold = df[numeric_field].mean() if df[numeric_field].notnull().any() else 0
        print(f"Threshold set to mean ({threshold:.2f}) of `{numeric_field}`.")
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std(ddof=0)
        print(f"\nNormalized `{numeric_field}` for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # For grouping, choose the first string/text field if available
        group_field = group_fields[0] if group_fields else None
        if group_field and group_field in df.columns:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"\nGrouped data by {group_field} (showing means of numeric fields):")
            print(grouped_df.head())
        else:
            print("No suitable group (categorical) field found for grouping.")
    else:
        print("No numeric field available for EDA.")
else:
    print("No tabular record sets available to analyze.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. For demonstration, we plot a histogram of the numeric field and a boxplot grouped by the group field when available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if chosen_record_set_id is not None and chosen_record_set_id in dataframes:
    df = dataframes[chosen_record_set_id]
    # Use same numeric/grouping fields as above
    rs_metadata = next(rs for rs in dataset.record_sets if rs['@id'] == chosen_record_set_id)
    numeric_fields = [f['@id'] for f in rs_metadata['fields'] if f.get('dataType', '').lower() in ['number', 'float', 'integer']]
    group_fields = [f['@id'] for f in rs_metadata['fields'] if f.get('dataType', '').lower() in ['string', 'text']]

    numeric_field = numeric_fields[0] if numeric_fields else None
    group_field = group_fields[0] if group_fields else None

    if numeric_field and numeric_field in df.columns:
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
        plt.figure(figsize=(8, 5))
        sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
        plt.title(f"Distribution of {numeric_field}")
        plt.xlabel(numeric_field)
        plt.ylabel('Frequency')
        plt.show()

        if group_field and group_field in df.columns:
            plt.figure(figsize=(10, 6))
            sns.boxplot(x=group_field, y=numeric_field, data=df)
            plt.title(f"Boxplot of {numeric_field} by {group_field}")
            plt.xticks(rotation=45)
            plt.show()
else:
    print("No numeric data to visualize.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- This notebook demonstrated how to use the `mlcroissant` library for loading Croissant datasets, exploring available record sets by their `@id`, and extracting data programmatically.
- We referenced all entities, including record sets and fields, using their `@id`, as required for reproducible, schema-level data handling.
- Using the dataset, we performed basic EDA and visualizations for the available numeric fields, which can be adapted to probe deeper into the data for downstream analyses (e.g., protein abundance and biomarker discovery).

For further analysis, consult the Croissant metadata for more details on experimental design, data interpretation, or advanced feature selection.